# Notes
- The test does not rely on GPU at all
- This is only a proof-of-concept lab.
- More optimized version will be written with Rust

# Basic detection

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n_object365.pt')

img_path = r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\rtbench\assets\images\20250828_192121.jpg"
res = model.predict(img_path, show=True, conf=0.2)
res[0].show()

# Extract model for CPU runtime on Rust
# model.export(
#     format="onnx",
#     simplify=True,
#     opset=17,
#     dynamic=False,
#     half=True,

#     # Output-specific params
#     imgsz=320,
#     max_det=50,
#     verbose=False,
# )

# Basic OCR

In [ ]:
import easyocr

reader = easyocr.Reader(['en', 'vi'], verbose=False, gpu=False, cudnn_benchmark=True)

img_path = r"C:\Users\Bui Thien Nghia\Pictures\Screenshot_20220307-132823.png"
results = reader.readtext(img_path, batch_size=4)
for bbox, text, confidence in results:
    if float(confidence) < 0.5: continue
    print(f'Text: {text} (Confidence: {confidence:.2f})')


# Fused

### Loading & Helper

In [1]:
import cv2
import numpy as np
import easyocr
import os
import subprocess
import platform
from time import time
from ultralytics import YOLO


def put_outlined_text(img, text, org, font, font_scale, thickness=2):
    x, y = org
    border_thickness = thickness + 2
    for dx, dy in [(-1, -1), (-1, 1), (1, -1), (1, 1), (0, -1), (0, 1), (-1, 0), (1, 0)]:
        cv2.putText(img, text, (x + dx * border_thickness, y + dy * border_thickness), 
                    font, font_scale, (0, 0, 0), border_thickness, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)


def adaptive_resize(image, max_dim: int = 1280):
    h, w = image.shape[:2]
    if max(h, w) <= max_dim:
        return image, 1.0
    if h > w:
        scale = max_dim / h
    else:
        scale = max_dim / w
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized_image, scale


model = YOLO('yolo11n_object365.pt')
reader = easyocr.Reader(['en', 'vi'], verbose=False, gpu=False, cudnn_benchmark=True)

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\.venv\Lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


### Running

In [ ]:
img_path = r"C:\Users\Bui Thien Nghia\Pictures\IMG_20250106_095948.jpg"
image = cv2.imread(img_path)
blocklist = None # Blocks unused character

# Runtime test individual
s = time()

results = reader.readtext(img_path, batch_size=4, blocklist=blocklist)
for bbox, text, confidence in results:
    if float(confidence) < 0.5: continue
    pts = np.array(bbox, dtype=np.int32).reshape((-1, 1, 2))
    cv2.polylines(image, [pts], isClosed=True, color=(0, 255, 0), thickness=2)
    top_left = tuple(map(int, bbox[0]))
    display_text = f"{text} ({confidence:.2f})"
    text_origin = (top_left[0], max(top_left[1] - 10, 15))
    put_outlined_text(image, display_text, text_origin, cv2.FONT_HERSHEY_SIMPLEX, font_scale=0.6, thickness=2)

e = time()
print(f"Time elapsed: {e - s:.3f}s")

output_path = img_path.split('\\')[-1]
cv2.imwrite(output_path, image)
print(f"Result saved to {output_path}. Opening in system gallery...")

absolute_path = os.path.abspath(output_path)
system_name = platform.system()

if system_name == 'Windows':
    os.startfile(absolute_path)
elif system_name == 'Darwin':  # macOS
    subprocess.run(['open', absolute_path])
else:  # Linux
    subprocess.run(['xdg-open', absolute_path])


res = model.predict(img_path, show=True, conf=0.2, device="cpu")
res[0].show()

c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Time elapsed: 1.795s
Result saved to IMG_20250106_095948.jpg. Opening in system gallery...

image 1/1 C:\Users\Bui Thien Nghia\Pictures\IMG_20250106_095948.jpg: 416x640 1 Person, 39.5ms
Speed: 1.1ms preprocess, 39.5ms inference, 0.6ms postprocess per image at shape (1, 3, 416, 640)


# Mass detection

In [ ]:
# TODO
# Write a code snippet that runs detection on the entire current database
# Results should be stored in a dictionary. The items should be SQLite-ready
# Each item should also have a string that is concatenated from all text detection and object's classes
# Do not create a new class. Only define new functions if necessary
# Measure the runtime, including mean, median, standard deviation, min, and max

import glob
import os
import time
import json
from tqdm import tqdm

# Define the image directory and retrieve all image files
image_dir = r"C:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\rtbench\assets\images"
image_paths = glob.glob(os.path.join(image_dir, "*.*"))

results_dict = {}
runtimes = []

pbar = tqdm(image_paths, colour="green", leave=False)
for img_path in pbar:
    try:
        file_name = os.path.basename(img_path)
        pbar.set_description(f"Processing image: {file_name}")
        start_time = time.time()

        image = cv2.imread(img_path)
        if image is None: continue
        resized, scale = adaptive_resize(image)
        
        # YOLO
        yolo_res = model.predict(img_path, verbose=False, conf=0.2, device="cpu")
        detected_classes = []
        object_boxes = []
        if yolo_res and len(yolo_res) > 0:
            names = yolo_res[0].names
            boxes = yolo_res[0].boxes
            for i, box in enumerate(boxes):
                cls_id = int(box.cls[0])
                cls_name = names[cls_id]
                detected_classes.append(cls_name)
                xyxy = box.xyxy[0].tolist()
                xyxy_original = [coord / scale for coord in xyxy]
                conf = float(box.conf[0])
                object_boxes.append({
                    "class": cls_name,
                    "confidence": conf,
                    "bbox": xyxy_original
                })
                
        # EasyOCR
        ocr_results = reader.readtext(img_path, batch_size=4)
        detected_texts = []
        ocr_boxes = []
        for bbox, text, confidence in ocr_results:
            if float(confidence) >= 0.5:
                detected_texts.append(text)
                original_bbox = [[int(float(pt[0]) / scale), int(float(pt[1]) / scale)] for pt in bbox]
                ocr_boxes.append({
                    "text": text,
                    "confidence": float(confidence),
                    "bbox": [[int(pt[0]), int(pt[1])] for pt in original_bbox]
                })
                
        end_time = time.time()
        elapsed = end_time - start_time
        runtimes.append(elapsed)
        
        combined_content = " ".join(detected_classes + detected_texts)
        results_dict[file_name] = {
            "file_path": img_path,
            "object_classes": ", ".join(detected_classes),
            "text_detections": ", ".join(detected_texts),
            "combined_text": combined_content,
            "yolo_detections": object_boxes,
            "ocr_detections": ocr_boxes,
            "runtime": elapsed
        }
    except Exception as e:
        print(f"Error while processing image {file_name}: {e}")
        results_dict[file_name] = {}

output_json_path = "detection_results.json"
with open(output_json_path, "w", encoding="utf-8") as f:
    json.dump(results_dict, f, ensure_ascii=False, indent=4)

if runtimes:
    mean_time = np.mean(runtimes)
    median_time = np.median(runtimes)
    stdev_time = np.std(runtimes) if len(runtimes) > 1 else 0.0
    min_time = np.min(runtimes)
    max_time = np.max(runtimes)
    
    print(f"Processed {len(runtimes)} images successfully.")
    print(f"Runtime Metrics -> Mean: {mean_time:.3f}s | Median: {median_time:.3f}s | StdDev: {stdev_time:.3f}s | Min: {min_time:.3f}s | Max: {max_time:.3f}s")

Processing image: 140A3607.JPG:   0%|          | 0/1000 [00:00<?, ?it/s]c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Processing image: 20250729_183751.jpg:   0%|          | 1/1000 [00:20<5:42:19, 20.56s/it]c:\Users\Bui Thien Nghia\Documents\PERSONAL FILES 2\VGU_CS\Y2\Intro CS\Opsis Gallery\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


error: OpenCV(5.0.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4217: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'
